In [15]:
from __future__ import annotations

from typing import Annotated, ClassVar

import numpy as np
from pydantic import BaseModel, Field

from mrg32k3a.mrg32k3a import MRG32k3a
from simopt import dsl
from simopt.base import (
    ConstraintType,
    Model,
    Problem,
    VariableType,
)
from simopt.input_models import Normal
# from simopt.simulations.example import (
#     Example2ModelConfig,
#     ExampleModelConfig,
#     replicate,
#     replicate_discrete,
# )
from simopt.utils import override

# Like in simulation 
class QuadNoiseConfig(BaseModel):
    x: Annotated[
        tuple[float, ...],
        Field(default=(5.0,), description="decision variable"),
    ]
    noise_std: Annotated[
        float,
        Field(default=1e-8, ge=0.0, description="tiny noise standard deviation"),
    ]

class QuadNoiseModel(Model[QuadNoiseConfig]):
    """x^2 + 1 evaluated with tiny stochastic noise."""

    class_name_abbr: ClassVar[str] = "QUADNOISE"
    class_name: ClassVar[str] = "x^2 + 1 with tiny noise"
    config_class: ClassVar[type[QuadNoiseConfig]] = QuadNoiseConfig
    n_rngs: ClassVar[int] = 1
    n_responses: ClassVar[int] = 1
    
    def __init__(self, fixed_factors: dict | None = None) -> None:
        super().__init__(fixed_factors)
        self.noise_model = Normal()
        
    def replicate(self, factors: QuadNoiseConfig, rngs: list[MRG32k3a]) -> tuple[dict, dict]:
        x = np.asarray(factors.x, dtype=float)
        f = float(np.sum(x**2) + 1.0)

        # Tiny noise: set noise_std = 0.0 for exact deterministic behavior.
        noise = self.noise_model.random(rngs[0], mu=0.0, sigma=factors.noise_std)
        noisy_value = f + noise

        responses = {"est_f(x)": noisy_value}
        gradients = {"est_f(x)": {"x": 2.0 * x}}
        return responses, gradients
    
    #Similar to example.py in models.   
class QuadNoiseProblemConfig(BaseModel):
    initial_solution: Annotated[
        tuple[float, ...],
        Field(default=(5.0,), description="initial point"),
    ]
    budget: Annotated[
        int,
        Field(default=1000, gt=0, description="max simulation budget"),
    ]
    
class QuadNoiseProblem(Problem):
    """Minimize x^2 + 1 under noisy stochastic replications."""

    class_name_abbr: ClassVar[str] = "QUADNOISE"
    class_name: ClassVar[str] = "Minimize x^2 + 1 with zero noise"
    config_class: ClassVar[type[BaseModel]] = QuadNoiseProblemConfig
    model_class: ClassVar[type[Model]] = QuadNoiseModel
    n_objectives: ClassVar[int] = 1
    n_stochastic_constraints: ClassVar[int] = 0
    minmax: ClassVar[tuple[int, ...]] = (-1,)
    constraint_type: ClassVar[ConstraintType] = ConstraintType.UNCONSTRAINED
    variable_type: ClassVar[VariableType] = VariableType.CONTINUOUS
    gradient_available: ClassVar[bool] = True
    model_default_factors: ClassVar[dict] = {}
    model_decision_factors: ClassVar[set[str]] = {"x"}
    
    @property
    def optimal_value(self) -> float | None:
        return 1.0
    
    @property
    def optimal_solution(self) -> tuple | None:
        return (0.0,)
    
    @override
    def build(self) -> dsl.Model:
        problem = dsl.Model()
        x0 = self.factors["initial_solution"]
        x = problem.add_continuous_vector(
            lb=-10.0, ub=10.0, shape=(len(x0),), initial=x0,
        )
        simulation = self.add_simulation(problem, {"x": x})
        problem.minimize(dsl.mean(simulation.metric("est_f(x)")))
        return problem
    
    def vector_to_factor_dict(self, vector: tuple) -> dict:
        return {"x": vector[:]}
    
    def get_random_solution(self, rand_sol_rng: MRG32k3a) -> tuple:
        # unchanged from the example
        return tuple(
            rand_sol_rng.mvnormalvariate(
                mean_vec=[0.0] * self.dim,
                cov=np.eye(self.dim).tolist(),
                factorized=False,
            )
        )

In [16]:
import numpy as np
import pandas as pd
from simopt.solvers.astrodf import ASTRODF
from simopt.experiment_base import ProblemSolver
# from your_file import QuadNoiseProblem   # import the problem class you defined

LOG = []
STATE = {"mrep": -1}


class LoggingASTRODF(ASTRODF):

    def _initialize_solving(self, ctx):
        super()._initialize_solving(ctx)
        STATE["mrep"] += 1

    def construct_model(self, ctx):
        delta_in = self.delta_k
        out = super().construct_model(ctx)
        fval, y_var, q, grad, hessian, solns = out

        m = 2 * self.problem.dim + 1
        for i, s in enumerate(solns[-m:]):
            LOG.append({
                "mrep": STATE["mrep"],
                "iter": self.iteration_count,
                "delta_in": delta_in,
                "incumbent_x": tuple(np.ravel(self.incumbent_x)),
                "point_idx": i,
                "x": tuple(np.ravel(s.x)),
                "F_hat": float(np.ravel(fval[i])[0]),
                "sample_size": int(s.n_reps),
                "var_F": float(np.ravel(s.objectives_var)[0]),
                "model_grad": tuple(np.ravel(grad)),
                "model_hessian_diag": tuple(np.ravel(hessian)),
            })
        return out


solver = LoggingASTRODF(fixed_factors={"use_gradients": False})
problem = QuadNoiseProblem(fixed_factors={"initial_solution": (1.0,), "budget": 1000})

exp = ProblemSolver(solver=solver, problem=problem)
exp.run(n_macroreps=1, n_jobs=1)

df = pd.DataFrame(LOG)
pd.set_option("display.width", 250, "display.max_rows", None, "display.max_columns", None)
print(df[["mrep", "iter", "delta_in", "point_idx", "x", "F_hat", "sample_size"]])
df.to_csv("astrodf_design_sets_quad.csv", index=False)

     mrep  iter      delta_in  point_idx                           x       F_hat  sample_size
0       0     1  1.000000e+01          0                      (1.0,)    2.000000            5
1       0     1  1.000000e+01          1                     (9.99,)  100.800100            5
2       0     1  1.000000e+01          2                     (-9.0,)   82.000000            5
3       0     2  5.000000e+00          0                      (1.0,)    2.000000            5
4       0     2  5.000000e+00          1     (-0.05145914347922664,)    1.002648            5
5       0     2  5.000000e+00          2                      (6.0,)   37.000000            5
6       0     3  2.500000e+00          0                      (1.0,)    2.000000            5
7       0     3  2.500000e+00          1      (-1.0000000000000013,)    2.000000            5
8       0     3  2.500000e+00          2                      (3.5,)   13.250000            5
9       0     4  1.250000e+00          0                    

In [12]:
import numpy as np
from numpy.linalg import norm
from scipy.optimize import NonlinearConstraint, minimize
from simopt.solvers.astrodf import ASTRODF


def _v(x):
    return np.asarray(x, dtype=float).ravel().tolist()


class DebugASTRODF(ASTRODF):

    def perform_adaptive_sampling(self, ctx, solution, pilot_run, delta_k,
                                  compute_kappa=False, label="sol"):
        sample_size = solution.n_reps if solution.n_reps > 0 else pilot_run
        lambda_max = ctx.budget.remaining
        n_start = solution.n_reps

        if solution.n_reps == 0:
            solution = ctx.evaluate(solution, pilot_run)
            sample_size = pilot_run

        reason = None
        while True:
            sig2 = solution.objectives_var[0]
            if self.delta_power == 0:
                sig2 = max(sig2, np.trace(solution.objectives_gradients_var))

            kappa = None
            if compute_kappa:
                if self.enable_gradient:
                    rhs = norm(solution.objectives_gradients_mean[0])
                else:
                    rhs = solution.objectives_mean
                kappa = (rhs * np.sqrt(pilot_run) / (delta_k ** (self.delta_power / 2))).item()

            if kappa is not None:
                k = kappa
            elif self.kappa is not None:
                k = self.kappa
            else:
                k = 0

            stopping = self.get_stopping_time(pilot_run, sig2, delta_k, k)

            if ctx.budget.remaining <= 0:
                reason = "budget exhausted (remaining <= 0)"
            elif sample_size >= min(stopping, lambda_max):
                if stopping <= lambda_max:
                    reason = "sample_size >= stopping time (variance rule satisfied)"
                else:
                    reason = "sample_size >= lambda_max (capped by remaining budget at entry)"

            if reason is not None:
                if compute_kappa:
                    self.kappa = kappa
                break

            solution = ctx.evaluate(solution, 1)
            sample_size += 1

        print(
            f"  [sampling:{label}] x={_v(solution.x)} "
            f"f_hat={_v(solution.objectives_mean)} "
            f"var_f={_v(solution.objectives_var)} "
            f"grad_hat={_v(solution.objectives_gradients_mean[0]) if self.enable_gradient else 'n/a'}"
        )
        print(
            f"                  n_reps_before={n_start} n_reps_after={solution.n_reps} "
            f"pilot_run={pilot_run} sig2={float(sig2):.6g} kappa_used={float(k):.6g} "
            f"delta_power={self.delta_power} delta_k={delta_k:.6g}"
        )
        print(
            f"                  stopping_time={stopping} lambda_max={lambda_max} "
            f"budget_remaining={ctx.budget.remaining}"
        )
        print(f"                  STOP REASON: {reason}")

    def iterate(self, ctx):
        self.iteration_count += 1
        print("\n" + "=" * 90)
        print(f"ITER={self.iteration_count}  budget_remaining={ctx.budget.remaining}")

        pilot_run = int(np.ceil(
            max(
                self.lambda_min * np.log10(10 + self.iteration_count) ** 1.1,
                min(0.5 * self.problem.dim, ctx.budget.total),
            )
            - 1
        ))

        if self.iteration_count == 1:
            self.incumbent_solution = ctx.evaluate(self.incumbent_x, 0)
            self.visited_pts_list.append(self.incumbent_solution)
            self.perform_adaptive_sampling(
                ctx, self.incumbent_solution, pilot_run, self.delta_k,
                compute_kappa=True, label="incumbent(init)",
            )
            ctx.log(self.incumbent_solution)
        elif self.factors["crn_across_solns"]:
            self.perform_adaptive_sampling(
                ctx, self.incumbent_solution, pilot_run, self.delta_k,
                label="incumbent",
            )

        if self.enable_gradient:
            fval = np.ones(2 * self.problem.dim + 1) * self.incumbent_solution.objectives_mean
            grad = self.incumbent_solution.objectives_gradients_mean[0]
            hessian = self.h_k
            q = np.array([])
            y_var = [[]]
            interpolation_solns = []
        else:
            (fval, y_var, q, grad, hessian, interpolation_solns) = self.construct_model(ctx)

        print(f"  incumbent_x={_v(self.incumbent_x)} f_hat(incumbent)={_v(fval[0])} "
              f"n_reps(incumbent)={self.incumbent_solution.n_reps}")
        print(f"  delta_k={self.delta_k:.6g} grad={_v(grad)} hessian={_v(hessian)}")
        if not self.enable_gradient:
            print(f"  interpolation f_hat={_v(fval)}")

        if self.easy_solve:
            dot_a = np.dot(grad, hessian) if self.enable_gradient else grad * hessian
            check_positive_definite = np.dot(dot_a, grad)

            if check_positive_definite <= 0:
                tau = 1.0
            else:
                tau = min(1.0, float(norm(grad) ** 3 / (self.delta_k * check_positive_definite)))

            grad = np.reshape(grad, (1, self.problem.dim))[0]
            grad_norm = norm(grad)
            if grad_norm == 0:
                candidate_x = self.incumbent_x
            else:
                candidate_x = self.incumbent_x - tau * self.delta_k * grad / float(grad_norm)
            print(f"  Cauchy step: tau={tau} ||grad||={float(grad_norm):.6g} "
                  f"g^T H g={_v(check_positive_definite)}")
        else:
            def subproblem(s):
                s_grad_dot = np.dot(s, grad)
                s_hessian_dot = np.dot(np.multiply(s, hessian), s)
                return float(np.ravel(fval[0] + s_grad_dot + s_hessian_dot)[0])

            def con_f(s):
                return float(norm(s))

            nlc = NonlinearConstraint(con_f, 0, self.delta_k)
            sol = minimize(subproblem, np.zeros(self.problem.dim),
                           method="trust-constr", constraints=nlc)
            candidate_x = self.incumbent_x + sol.x

        candidate_x = tuple(float(x) for x in candidate_x)

        candidate_solution = ctx.evaluate(candidate_x, 0)
        self.visited_pts_list.append(candidate_solution)

        if self.factors["crn_across_solns"]:
            num_sims = self.incumbent_solution.n_reps
            candidate_solution = ctx.evaluate(candidate_solution, num_sims)
            print(f"  [sampling:candidate] CRN: forced n_reps = incumbent n_reps = {num_sims} "
                  f"(no adaptive stopping rule applied)")
        else:
            self.perform_adaptive_sampling(
                ctx, candidate_solution, pilot_run, self.delta_k, label="candidate"
            )

        fval_tilde = candidate_solution.objectives_mean
        print(f"  candidate_x={_v(candidate_x)} f_hat(candidate)={_v(fval_tilde)} "
              f"n_reps(candidate)={candidate_solution.n_reps}")

        if not self.enable_gradient:
            min_fval = min(fval)
            sufficient_reduction = (
                (fval[0] - min_fval)
                >= self.factors["ps_sufficient_reduction"] * self.delta_k ** 2
            )
            condition_met = min_fval < fval_tilde and sufficient_reduction

            high_variance = False
            if not condition_met and candidate_solution.objectives_mean[0] != 0:
                high_variance = (
                    candidate_solution.objectives_var[0]
                    / (candidate_solution.n_reps * candidate_solution.objectives_mean[0] ** 2)
                ) > 0.75

            print(f"  pattern search: min_fval={_v(min_fval)} "
                  f"sufficient_reduction={bool(np.all(sufficient_reduction))} "
                  f"condition_met={bool(np.all(condition_met))} high_variance={high_variance}")

            if condition_met or high_variance:
                fval_tilde = min_fval
                min_idx = np.argmin(fval)
                candidate_x = y_var[min_idx][0]
                candidate_solution = interpolation_solns[min_idx]
                print(f"  -> replaced candidate by interpolation point {_v(candidate_x)}")

        s = np.subtract(np.array(candidate_x), np.array(self.incumbent_x))

        if self.enable_gradient:
            model_reduction = -np.dot(s, grad) - 0.5 * np.dot(np.dot(s, hessian), s)
        else:
            model_reduction = (
                self.evaluate_model(np.zeros(self.problem.dim), q)
                - self.evaluate_model(s, q)
            )

        actual_reduction = fval[0] - fval_tilde
        rho = 0 if np.all(model_reduction <= 0) else actual_reduction / model_reduction

        print(f"  actual_reduction={_v(actual_reduction)} model_reduction={_v(model_reduction)} "
              f"rho={_v(rho)} (eta_1={self.eta_1}, eta_2={self.eta_2})")

        successful = bool(np.all(rho >= self.eta_1))
        if successful:
            self.incumbent_x = candidate_x
            self.incumbent_solution = candidate_solution
            ctx.log(candidate_solution)
            self.delta_k = min(self.delta_k, self.delta_max)
            outcome = "successful (accept)"
            if np.all(rho >= self.eta_2):
                self.delta_k = min(self.gamma_1 * self.delta_k, self.delta_max)
                outcome = "very successful (accept, expand)"
            if self.enable_gradient:
                self.update_hessian(candidate_solution, grad, s)
        else:
            self.delta_k = min(self.gamma_2 * self.delta_k, self.delta_max)
            outcome = "unsuccessful (reject, shrink)"

        print(f"  OUTCOME: {outcome} -> new incumbent_x={_v(self.incumbent_x)} "
              f"new delta_k={self.delta_k:.6g} delta_max={self.delta_max:.6g}")

In [14]:
exp = ProblemSolver(
    solver=DebugASTRODF(fixed_factors={"use_gradients": True}),
    problem=QuadNoiseProblem(
        fixed_factors={"initial_solution": (1.0,), "budget": 1000},
    ),
)

exp.run(n_macroreps=1)
exp.post_replicate(n_postreps=20)


ITER=1  budget_remaining=1000
  [sampling:incumbent(init)] x=[1.0] f_hat=[1.999999999564376] var_f=[0.0] grad_hat=[2.0]
                  n_reps_before=0 n_reps_after=5 pilot_run=5 sig2=0 kappa_used=4.47214 delta_power=0 delta_k=10
                  stopping_time=5 lambda_max=1000 budget_remaining=995
                  STOP REASON: sample_size >= stopping time (variance rule satisfied)
  incumbent_x=[1.0] f_hat(incumbent)=[1.999999999564376] n_reps(incumbent)=5
  delta_k=10 grad=[2.0] hessian=[1.0]
  Cauchy step: tau=0.2 ||grad||=2 g^T H g=[4.0]
  [sampling:candidate] CRN: forced n_reps = incumbent n_reps = 5 (no adaptive stopping rule applied)
  candidate_x=[-1.0] f_hat(candidate)=[1.999999999564376] n_reps(candidate)=5
  actual_reduction=[0.0] model_reduction=[2.0] rho=[0.0] (eta_1=0.1, eta_2=0.8)
  OUTCOME: unsuccessful (reject, shrink) -> new incumbent_x=[1.0] new delta_k=5 delta_max=6.89143

ITER=2  budget_remaining=990
  [sampling:incumbent] x=[1.0] f_hat=[1.999999999564376] var